# Week 4 Protocol: Difficulty Calibration v2 (Dev Set Only)

<a href="https://colab.research.google.com/github/jorshimayor/AI-Engineering-Flow_Research/blob/main/jupyter/week4_calibration_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Why a second calibration.** Calibration v1 (results in `results/week4_dev_v1/`) found no level that suited both models, and showed a flaw: shuffling *every* sentence made the text incoherent (1.5B solved 71% of level 2 problems when the scene sentence happened to stay first, 15% otherwise). Protocol v1.2 fixes the generator ([`src/react_tasks_v2.py`](../src/react_tasks_v2.py)): the scene-setting sentence always comes first, and there are four levels:

| Level | Steps | Distractors | Order |
|---|---|---|---|
| 1 | 3 | 1 | natural (unchanged from v1, so its v1 results are reused) |
| 2 | 3 | 1 | scene first, rest shuffled |
| 3 | 3 | 2 | scene first, rest shuffled |
| 4 | 4 | 2 | scene first, rest shuffled |

**Selection rule (protocol 3.2.1, written before this run):**
1. Prefer one common level: Act-only between 40% and 85% for both models, closest to 62.5%.
2. If no common level exists, each model gets its own level: the in-band level closest to 62.5% for that model.
3. A model with no in-band level fails gate 5.

Then direct and ReAct run on dev at each model's chosen level to check the other validity gates.

It only ever touches the **dev** split. The 400 test problems stay unseen until the protocol is frozen.

**Before you start:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`. Roughly 60 to 90 minutes on a free T4 (levels 2 to 4 only; level 1 is reused) (an estimate: the harder problems are longer than Week 3's). Results are saved one problem at a time, so a disconnect only costs the problem in progress.

## 1. Check the GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → T4 GPU, then run again."
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB")

## 2. Get the code

In [ ]:
REPO = "https://github.com/jorshimayor/AI-Engineering-Flow_Research.git"
BRANCH = "main"
import os
# Always use the latest code: clone if missing, otherwise update in place (a reused runtime keeps the old folder)
if not os.path.exists("AI-Engineering-Flow_Research"):
    !git clone --depth 1 --branch {BRANCH} {REPO}
%cd /content/AI-Engineering-Flow_Research
!git fetch -q --depth 1 origin {BRANCH} && git checkout -q -B {BRANCH} FETCH_HEAD
!pip install -q "transformers==4.49.0" accelerate
!git log --oneline -1

## 3. Keep results on Google Drive (recommended)

In [ ]:
USE_DRIVE = True
RESULTS = "results/week4_dev_v2"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    target = "/content/drive/MyDrive/AI-Engineering/week4_dev_v2"
    os.makedirs(target, exist_ok=True); os.makedirs("results", exist_ok=True)
    if not os.path.islink(RESULTS):
        if os.path.isdir(RESULTS):
            os.rmdir(RESULTS)
        os.symlink(target, RESULTS)
os.makedirs(RESULTS, exist_ok=True)
# Level 1 is identical to calibration v1, so reuse those runs instead of repeating them
import shutil, glob
for f in glob.glob("results/week4_dev_v1/*-L1.jsonl"):
    dst = os.path.join(RESULTS, os.path.basename(f))
    if not os.path.exists(dst):
        shutil.copy(f, dst); print("reused", os.path.basename(f))
print("results go to:", os.path.realpath(RESULTS))

## 4. Check the generator

Automatic checks on every split and level: answers equal their solution expressions, answers are positive integers, the step count matches the level, and no test template shares an exemplar's operations (gate 4). Test questions are checked but never printed.

In [ ]:
import sys, json, gc
from pathlib import Path
sys.path.insert(0, "src")
import importlib, react_tasks_v2
importlib.reload(react_tasks_v2)              # in case an older version was imported earlier in this session
from react_tasks_v2 import make_problems, check_problems, leakage_check, GENERATOR_VERSION
assert GENERATOR_VERSION in ("1.2", "1.3"), f"stale code (generator {GENERATOR_VERSION}): Runtime → Disconnect and delete runtime, then Run all"
print("generator version", GENERATOR_VERSION)
from react_tasks_v2 import LEVELS
for split in ("dev", "test"):
    for L in LEVELS:
        c = check_problems(make_problems(split, L))
        print(f"{split:4} level {L}: {c['n']:3} problems, errors {len(c['errors'])}")
        assert not c["errors"], c["errors"][:3]
issues = leakage_check()
print("leakage:", issues or "none"); assert not issues
print("\nOne dev problem per level:")
for L in LEVELS:
    p = make_problems("dev", L)[0]; print(f"[L{L}] {p['question']}  -> {p['gold']:,}")

## 5. Stage A: Act-only on dev at all four levels

Each model is loaded once and freed before the next.

In [ ]:
from react_agent import LocalLLM
from run_react_bench import run, failure_type, is_format_failure
from stats import wilson_ci
import torch

MODELS = ["Qwen/Qwen2.5-1.5B-Instruct", "Qwen/Qwen2.5-3B-Instruct"]
LEVELS = [1, 2, 3, 4]
MAX_STEPS = 8
def out_path(model, level): return Path(RESULTS) / f"{model.split('/')[-1]}-L{level}.jsonl"

for model in MODELS:
    llm = LocalLLM(model)
    for L in LEVELS:
        run(model, ["act"], make_problems("dev", L), out_path(model, L), MAX_STEPS, llm=llm)
    del llm; gc.collect(); torch.cuda.empty_cache()

## 6. Apply the selection rule (protocol 3.2.1)

In [ ]:
def load(model, level):
    f = out_path(model, level)
    return [json.loads(l) for l in f.open()] if f.exists() else []

def acc(rows, mode):
    rs = [r for r in rows if r["mode"] == mode]
    return (sum(r["correct"] for r in rs) / len(rs), len(rs)) if rs else (float("nan"), 0)

table = {}
print(f"{'level':6}" + "".join(f"{m.split('-')[1]:>22}" for m in MODELS) + "   eligible?")
for L in LEVELS:
    accs = [acc(load(m, L), "act") for m in MODELS]
    ok = all(0.40 <= a <= 0.85 and n == 40 for a, n in accs)
    table[L] = dict(act={m: a for m, (a, _) in zip(MODELS, accs)}, eligible=ok)
    cells = "".join(f"{a:8.1%} (CI {wilson_ci(round(a*n), n)[0]:.0%} to {wilson_ci(round(a*n), n)[1]:.0%})" for a, n in accs)
    print(f"{L:<6}{cells}   {'yes' if ok else 'no'}")

eligible = [L for L in LEVELS if table[L]["eligible"]]
in_band = lambda m, L: 0.40 <= table[L]["act"][m] <= 0.85
if eligible:
    common = min(eligible, key=lambda L: abs(sum(table[L]["act"].values()) / len(MODELS) - 0.625))
    CHOSEN = {m: common for m in MODELS}; MODE = "common"
    print(f"\nCommon level: {common}  (eligible: {eligible})")
else:
    CHOSEN, MODE = {}, "per-model"
    for m in MODELS:
        ok = [L for L in LEVELS if in_band(m, L)]
        CHOSEN[m] = min(ok, key=lambda L: abs(table[L]["act"][m] - 0.625)) if ok else None
    print("\nNo common level. Per-model levels (rule step 2):", {m.split('/')[-1]: L for m, L in CHOSEN.items()})
    for m, L in CHOSEN.items():
        if L is None: print(f"   {m}: no in-band level, gate 5 fails for this model")

## 7. Stage B: Direct and ReAct on dev at the chosen level

Runs at each model's chosen level. Checks gate 1 (direct ≤ 10%) and gate 3 (format failures ≤ 20% in both modes) before freezing.

In [ ]:
for model in MODELS:
    if CHOSEN.get(model) is None:
        continue
    llm = LocalLLM(model)
    run(model, ["direct", "react"], make_problems("dev", CHOSEN[model]), out_path(model, CHOSEN[model]), MAX_STEPS, llm=llm)
    del llm; gc.collect(); torch.cuda.empty_cache()

## 8. Gate report and calibration record

In [ ]:
from collections import Counter
report = dict(protocol_version="1.2", calibration="v2", split="dev", levels=LEVELS, selection_mode=MODE, chosen_levels=CHOSEN, selection_table=table, gates={})
for model in MODELS:
    if CHOSEN.get(model) is None:
        report["gates"][model] = dict(passed={"5_act_in_40_85": False}); print(f"\n{model}: no in-band level, gate 5 FAIL"); continue
    rows = load(model, CHOSEN[model])
    a_act, _ = acc(rows, "act"); a_react, _ = acc(rows, "react"); a_dir, _ = acc(rows, "direct")
    ff = {m: sum(is_format_failure(r) for r in rows if r["mode"] == m) / max(1, sum(r["mode"] == m for r in rows)) for m in ("act", "react")}
    gates = {"1_direct_le_10pct": a_dir <= 0.10, "3_format_fail_le_20pct": all(v <= 0.20 for v in ff.values()),
             "5_act_in_40_85": 0.40 <= a_act <= 0.85}
    report["gates"][model] = dict(direct=a_dir, act=a_act, react=a_react, format_fail=ff, passed=gates)
    print(f"\n{model} (level {CHOSEN[model]}): direct {a_dir:.0%} | Act {a_act:.0%} | ReAct {a_react:.0%} | format failures Act {ff['act']:.0%}, ReAct {ff['react']:.0%}")
    for g, ok in gates.items(): print(f"   gate {g}: {'PASS' if ok else 'FAIL'}")
    print("   ReAct failure types:", dict(Counter(failure_type(r) for r in rows if r['mode'] == 'react' and not r['correct'])))
report["all_gates_passed"] = all(all(v["passed"].values()) for v in report["gates"].values())
print("\nALL GATES PASSED" if report["all_gates_passed"] else "\nSome gates failed: share the results before freezing.")
json.dump(report, open(Path(RESULTS) / "calibration.json", "w"), indent=1, default=str)
print("saved", Path(RESULTS) / "calibration.json")

## 9. Save the results

In [ ]:
!cd results && zip -qr /content/week4_dev_v2_calibration.zip week4_dev_v2/
from google.colab import files
files.download("/content/week4_dev_v2_calibration.zip")

Send the zip back (or unzip it into `results/week4_dev_v2/` in the repo). Next: record the chosen level in the protocol, freeze it with a commit, then run the 400 test problems.